In [20]:
import pandas as pd
import numpy as np
from scipy import stats
#from scipy.stats import ttest_rel
import itertools

In [ ]:
df = pd.read_csv("novalite_output.csv")
df["HCV_active"] = df["HCV"].isin(["acute", "chronic"]).astype(int)

In [37]:
measures = [
    "Withdrawal", "Public_prop", "Recept_sharing", "Partners",
    "Drug_out", "Partner_HCV_prop", "Treatment",
    "Injections", "Alt_route", "HCV_active"
]

results = []

for v in measures:
    agg = df.groupby(["Agent", "Homeless"])[v].mean().reset_index()
    wide = agg.pivot(index="Agent", columns="Homeless", values=v)
    wide.columns = ["stable", "homeless"]
    wide = wide.dropna()
    diff = wide["homeless"] - wide["stable"]
    smd = diff.mean() / (diff.std(ddof=1) + 1e-8)

    results.append({
        "Variable": v,
        "Difference": diff.mean(),
        "Std_mean_diff": smd
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Std_mean_diff", ascending=False)

print(results_df)

           Variable  Difference  Std_mean_diff
1       Public_prop    0.321607       1.121978
9        HCV_active    0.017857       0.134539
2    Recept_sharing    0.006384       0.023504
7        Injections    0.013393       0.018493
6         Treatment    0.000000       0.000000
5  Partner_HCV_prop    0.000000       0.000000
8         Alt_route    0.000000       0.000000
3          Partners   -0.017857      -0.056945
4          Drug_out   -0.810848      -0.070039
0        Withdrawal   -0.123571      -0.080444


In [23]:
def paired_bootstrap(df, outcome_col, agent_col="Agent",
                     condition_col="Homeless",
                     n_boot=1000,
                     seed=42):
    np.random.seed(seed)

    wide = df.pivot(index=agent_col,
                    columns=condition_col,
                    values=outcome_col).dropna()

    agents = wide.index.values
    diffs = []

    for _ in range(n_boot):
        sampled_agents = np.random.choice(agents, size=len(agents), replace=True)
        sample = wide.loc[sampled_agents]
        diffs.append((sample[1] - sample[0]).mean())

    diffs = np.array(diffs)
    point_est = (wide[1] - wide[0]).mean()

    return {
        "estimate": point_est,
        "ci_lower": np.percentile(diffs, 2.5),
        "ci_upper": np.percentile(diffs, 97.5),
        "bootstrap_sd": diffs.std(ddof=1)
    }

In [38]:
results = []

for v in measures:
    result = paired_bootstrap(df, outcome_col=v)

    agg = df.groupby(["Agent", "Homeless"])[v].mean().reset_index()
    wide = agg.pivot(index="Agent", columns="Homeless", values=v)
    wide.columns = ["stable", "homeless"]
    wide = wide.dropna()
    diff = wide["homeless"] - wide["stable"]
    smd = diff.mean() / (diff.std(ddof=1) + 1e-8)

    results.append({
        "Variable": v,
        "Difference": result["estimate"],
        "CI_lower": result["ci_lower"],
        "CI_upper": result["ci_upper"],
        "Std_mean_diff": smd
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Std_mean_diff", ascending=False)

print(results_df)

           Variable  Difference  CI_lower  CI_upper  Std_mean_diff
1       Public_prop    0.321607  0.286780  0.357724       1.121978
9        HCV_active    0.017857  0.004464  0.040179       0.134539
2    Recept_sharing    0.006384 -0.028584  0.044302       0.023504
7        Injections    0.013393 -0.080357  0.120536       0.018493
6         Treatment    0.000000  0.000000  0.000000       0.000000
5  Partner_HCV_prop    0.000000  0.000000  0.000000       0.000000
8         Alt_route    0.000000  0.000000  0.000000       0.000000
3          Partners   -0.017857 -0.058036  0.022321      -0.056945
4          Drug_out   -0.810848 -2.495817  0.392441      -0.070039
0        Withdrawal   -0.123571 -0.347340  0.058927      -0.080444


In [39]:
results_df.to_csv("outputs/nova_results_table.csv", index=False, encoding="utf-8")

In [40]:
table = pd.crosstab(df["Neighborhood"], df["Homeless"])
print(pd.crosstab(df["Neighborhood"], df["Homeless"], normalize="columns")) 

Homeless             0         1
Neighborhood                    
central       0.303571  0.058036
south side    0.205357  0.250000
west side     0.491071  0.691964


In [41]:
table = pd.crosstab(df["Race"], df["Neighborhood"], normalize="index")
print("\nNeighborhood distribution by Race (row %)")
print(table)


Neighborhood distribution by Race (row %)
Neighborhood   central  south side  west side
Race                                         
Hispanic      0.223214    0.000000   0.776786
NHBlack       0.000000    0.910714   0.089286
NHWhite       0.205357    0.000000   0.794643
Other         0.294643    0.000000   0.705357


In [42]:
def safe_corr(x, y):
    x = x.dropna()
    y = y.dropna()

    if len(x) < 2 or len(y) < 2:
        return np.nan
    if x.nunique() < 2 or y.nunique() < 2:
        return np.nan

    return x.corr(y)

results = []

for a, b in itertools.combinations(measures, 2):
    corr = safe_corr(df[a], df[b])
    n = df[[a, b]].dropna().shape[0]

    results.append({
        "var1": a,
        "var2": b,
        "correlation": corr,
        "abs_correlation": abs(corr) if pd.notna(corr) else np.nan
    })

corr_df = pd.DataFrame(results)
threshold = 0.3  

strong_corr = corr_df[
    (corr_df["abs_correlation"] >= threshold)
].sort_values(by="abs_correlation", ascending=False)

strong_corr = strong_corr.reset_index(drop=True)
print(strong_corr)

               var1            var2  correlation  abs_correlation
0  Partner_HCV_prop      HCV_active     0.884246         0.884246
1    Recept_sharing      Injections     0.555688         0.555688
2    Recept_sharing        Drug_out     0.436966         0.436966
3        Withdrawal  Recept_sharing     0.436274         0.436274
4       Public_prop  Recept_sharing     0.391047         0.391047
5        Withdrawal     Public_prop     0.378959         0.378959
6        Withdrawal      Injections     0.347392         0.347392
7       Public_prop        Partners     0.331407         0.331407
8        Withdrawal        Partners     0.307620         0.307620


In [29]:
baseline_vars = {
    "prior_injection": "Prev_Injection",
    "prior_sharing": "Prev_recept_sharing",
    "prior_network": "Prev_friend_entry",
}
output_vars = {
    "prior_injection": "Injections",
    "prior_sharing": "Recept_sharing",
    "prior_network": "Partner_HCV_prop",
}

def cohens_d(x0, x1):
    x0 = x0.dropna()
    x1 = x1.dropna()

    if len(x0) < 2 or len(x1) < 2:
        return np.nan

    s0, s1 = x0.std(ddof=1), x1.std(ddof=1)
    pooled = np.sqrt((s0**2 + s1**2) / 2)

    if pooled == 0 or np.isnan(pooled):
        return np.nan

    return (x1.mean() - x0.mean()) / pooled

results = []

for key in baseline_vars:
    base = baseline_vars[key]
    out = output_vars[key]

    sub = df[[base, out, "Homeless"]].dropna().copy()
    sub["change"] = sub[out] - sub[base]

    stable = sub[sub["Homeless"] == 0]["change"]
    homeless = sub[sub["Homeless"] == 1]["change"]

    d = cohens_d(stable, homeless)

    results.append({
        "relationship": f"{base} → {out}",
        "std_mean_change_effect": d
    })

result = pd.DataFrame(results)
print(result)

                           relationship  std_mean_change_effect
0           Prev_Injection → Injections                0.007558
1  Prev_recept_sharing → Recept_sharing                0.000735
2  Prev_friend_entry → Partner_HCV_prop                0.000000


In [30]:
def cohens_d(x0, x1):
    x0 = x0.dropna()
    x1 = x1.dropna()

    n0, n1 = len(x0), len(x1)
    if n0 < 2 or n1 < 2:
        return np.nan

    s0, s1 = x0.std(ddof=1), x1.std(ddof=1)
    pooled = np.sqrt(((n0 - 1)*s0**2 + (n1 - 1)*s1**2) / (n0 + n1 - 2))

    if pooled == 0 or np.isnan(pooled):
        return np.nan

    return (x1.mean() - x0.mean()) / pooled

In [34]:
df = df.copy()
df["experience"] = df["Age"] - df["Age_Started"]
df["experience_group"] = np.where(df["experience"] >= 3, "Experienced", "Not experienced")
results = []

for y in measures:
    exp0 = df[df["experience_group"] == "Not experienced"]
    exp1 = df[df["experience_group"] == "Experienced"]

    d_not_exp = cohens_d(
        exp0[exp0["Homeless"] == 0][y],
        exp0[exp0["Homeless"] == 1][y]
    )
    d_exp = cohens_d(
        exp1[exp1["Homeless"] == 0][y],
        exp1[exp1["Homeless"] == 1][y]
    )
    results.append({
        "Outcome": y,
        "Difference (Experienced - Not)": d_exp - d_not_exp
    })

experience_results = pd.DataFrame(results).round(4)
print(experience_results)

            Outcome  Difference (Experienced - Not)
0        Withdrawal                          0.1123
1       Public_prop                          0.0446
2    Recept_sharing                          0.0039
3          Partners                          0.0347
4          Drug_out                         -0.1042
5  Partner_HCV_prop                          0.0000
6         Treatment                             NaN
7        Injections                          0.0070
8         Alt_route                             NaN
9        HCV_active                          0.1041


In [ ]:
df["age_group"] = np.where(df["Age"] >= df["Age"].median(), "Old", "Young")
results = []

for y in measures:
    young_stable = df[(df["age_group"] == "Young") & (df["Homeless"] == 0)][y].dropna()
    young_homeless = df[(df["age_group"] == "Young") & (df["Homeless"] == 1)][y].dropna()

    old_stable = df[(df["age_group"] == "Old") & (df["Homeless"] == 0)][y]
    old_homeless = df[(df["age_group"] == "Old") & (df["Homeless"] == 1)][y]

    young_d = cohens_d(young_stable, young_homeless)
    old_d = cohens_d(old_stable, old_homeless)

    results.append({
        "Outcome": y,
        #"Cohens_d (Young)": young_d,
        #"Cohens_d (Old)": old_d,
        "Difference (Old - Young)": old_d - young_d if pd.notna(old_d) and pd.notna(young_d) else np.nan
    })

age_results = pd.DataFrame(results).round(3)
print(age_results)

            Outcome  Difference (Old - Young)
0        Withdrawal                    -0.163
1       Public_prop                    -0.438
2    Recept_sharing                    -0.010
3          Partners                    -0.049
4          Drug_out                     0.051
5  Partner_HCV_prop                     0.000
6         Treatment                       NaN
7        Injections                    -0.007
8         Alt_route                       NaN
9        HCV_active                    -0.005


In [ ]:
results = []
for y in measures:
    sub = df[["Gender", "Homeless", y]].dropna()

    female_stable = sub[(sub["Gender"] == "Female") & (sub["Homeless"] == 0)][y]
    female_homeless = sub[(sub["Gender"] == "Female") & (sub["Homeless"] == 1)][y]

    male_stable = sub[(sub["Gender"] == "Male") & (sub["Homeless"] == 0)][y]
    male_homeless = sub[(sub["Gender"] == "Male") & (sub["Homeless"] == 1)][y]

    female_d = cohens_d(female_stable, female_homeless)
    male_d = cohens_d(male_stable, male_homeless)

    results.append({
        "Outcome": y,
        #"Cohens_d (Female)": female_d,
        #"Cohens_d (Male)": male_d,
        "Difference (Female - Male)": female_d - male_d if pd.notna(female_d) and pd.notna(male_d) else np.nan
    })

gender_results = pd.DataFrame(results).round(4)
print(gender_results)

            Outcome  Difference (Female - Male)
0        Withdrawal                     -0.0312
1       Public_prop                     -0.0260
2    Recept_sharing                     -0.0056
3          Partners                      0.0088
4          Drug_out                     -0.0715
5  Partner_HCV_prop                      0.0000
6         Treatment                         NaN
7        Injections                     -0.0070
8         Alt_route                         NaN
9        HCV_active                      0.1311


In [32]:
results = []

for y in measures:
    for race, g in df.groupby("Race"):
        stable = g[g["Homeless"] == 0][y]
        homeless = g[g["Homeless"] == 1][y]
        d = cohens_d(stable, homeless)

        results.append({
            "Outcome": y,
            "Race": race,
            "Cohens_d": d
        })
race_d_df = pd.DataFrame(results)

race_summary = race_d_df.groupby("Outcome")["Cohens_d"].agg(
    race_range=lambda x: x.max() - x.min(),
    mean="mean"
).reset_index()

#race_d_df = pd.DataFrame(results)
#print(race_d_df)
print(race_summary)

            Outcome  race_range      mean
0         Alt_route         NaN       NaN
1          Drug_out    0.218636 -0.099529
2        HCV_active    0.101279  0.058482
3        Injections    0.015251 -0.000391
4  Partner_HCV_prop    0.000000  0.000000
5          Partners    0.140400 -0.012125
6       Public_prop    1.013338  1.235630
7    Recept_sharing    0.015706 -0.000471
8         Treatment         NaN       NaN
9        Withdrawal    0.148944 -0.040779
